# Passive DNS Flux Detection

Measure address churn, geographic spread, and TTL behavior in synthetic passive-DNS observations.

**Safety and scope:** This notebook uses deterministic synthetic data and makes no network requests. Its results are analytical leads, not attribution or identity claims.

## Goal

Prioritize domains with fast-changing resolution patterns for defensive investigation.


## Setup

The workflow runs offline with NumPy and Pandas. Parameters and source-like fields are visible so the analysis can be reviewed and rerun.

### Key Assumptions

- All records are synthetic and contain no real people or infrastructure.
- Scores prioritize review; they do not prove ownership, intent, identity, or location.
- Real use requires documented authority, provenance, source terms, and retention limits.


In [1]:
import numpy as np
import pandas as pd

SEED = 88
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 130)
pd.set_option("display.max_colwidth", 70)


## Steps

### 1. Create bounded synthetic observations


In [2]:
rows = []
for domain_index in range(15):
    for day in range(14):
        burst = int(domain_index in {2, 7, 11})
        rows.append({
            "domain": f"service-{domain_index:02d}.test",
            "day": day,
            "unique_ips": int(rng.poisson(1.5 + 3.5 * burst) + 1),
            "countries": int(rng.integers(1, 3 + 4 * burst)),
            "ttl_seconds": int(rng.choice([60, 120, 300] if burst else [300, 900, 3600])),
            "address_changes": int(rng.poisson(0.7 + 3.0 * burst)),
        })
passive_dns = pd.DataFrame(rows)
print(passive_dns.head(8).to_string(index=False))


         domain  day  unique_ips  countries  ttl_seconds  address_changes
service-00.test    0           1          2         3600                1
service-00.test    1           2          2          300                0
service-00.test    2           1          2          300                0
service-00.test    3           3          2         3600                0
service-00.test    4           4          1          300                1
service-00.test    5           2          1         3600                0
service-00.test    6           1          1          900                0
service-00.test    7           3          2          900                0


### 2. Analyze and rank the observations


In [3]:
domain_behavior = passive_dns.groupby("domain", as_index=False).agg(
    mean_ips=("unique_ips", "mean"),
    max_countries=("countries", "max"),
    median_ttl=("ttl_seconds", "median"),
    total_changes=("address_changes", "sum"),
)
domain_behavior["flux_score"] = (
    0.30 * np.minimum(domain_behavior["mean_ips"] / 6, 1)
    + 0.25 * np.minimum(domain_behavior["max_countries"] / 6, 1)
    + 0.30 * np.minimum(domain_behavior["total_changes"] / 45, 1)
    + 0.15 * (domain_behavior["median_ttl"] <= 120)
).round(3)
ranked_domains = domain_behavior.sort_values("flux_score", ascending=False)
print(ranked_domains.head(7).to_string(index=False))


         domain  mean_ips  max_countries  median_ttl  total_changes  flux_score
service-02.test  6.500000              6       120.0             68       1.000
service-07.test  6.214286              6       120.0             50       1.000
service-11.test  7.428571              6       120.0             51       1.000
service-12.test  2.642857              2       900.0             18       0.335
service-13.test  2.571429              2       900.0             13       0.299
service-05.test  2.571429              2       300.0             10       0.279
service-06.test  2.714286              2       900.0              9       0.279


## Checks

Run deterministic integrity and reasonableness checks.


In [4]:
assert len(passive_dns) == 210
assert ranked_domains["flux_score"].between(0, 1).all()
assert ranked_domains.iloc[0]["total_changes"] >= ranked_domains["total_changes"].median()
print("Checks passed; results identify behavioral outliers for corroboration.")


Checks passed; results identify behavioral outliers for corroboration.


## Next Steps

- Add provider baselines to reduce CDN false positives.
- Track source coverage and observation gaps by day.
